In [2]:
import pandas as pd
import plotly.io as pio
import plotly.express as px


"""
This code uses a data set containing 6820 movies (graph uses 5559 movies), spanning from 1986-2020.
Contains tons of values, ranging from:
 
budget, company, country, director, genre, gross,
name, rating, released, runtime, score, votes, star, writer, and year.

Then adjusts budget and gross earnings for inflation (to 2020)
Then creates a graph of the adjusted budget vs gross for each movie. 

movie data set: https://www.kaggle.com/datasets/danielgrijalvas/movies/data
inflation data set: https://www.kaggle.com/datasets/varpit94/us-inflation-data-updated-till-may-2021
"""

# Movie and inflation data frame created from sources above
df_movies = pd.read_csv(r"C:\Users\hidek\Downloads\Coding\datasets\movies.csv")
df_cpi = pd.read_csv(r"C:\Users\hidek\Downloads\Coding\datasets\US CPI.csv")

# Removes movies that have less than 10k votes on IMDB
df_movies = df_movies[df_movies["votes"] > 10000]

# Removes "Yearmon" and adds a dedicated year column and averages CPI to the year. Then caps to 2020
df_cpi["year"] = pd.to_datetime(df_cpi["Yearmon"]).dt.year
df_cpi = df_cpi.groupby("year")["CPI"].mean().reset_index()
df_cpi = df_cpi[df_cpi["year"] <= 2020]

# Merges movies and CPI on year
df_movies_adj = pd.merge(df_movies, df_cpi, on="year", how="inner")

# Filter zero budgets and gross
df_movies_adj = df_movies_adj[df_movies_adj["budget"] > 0]
df_movies_adj = df_movies_adj[df_movies_adj["gross"] > 0]

# 258.811167 is the average CPI for 2020, used as the base year
df_movies_adj["adj_budget"] = df_movies_adj["budget"] * (258.811167 / df_movies_adj["CPI"])
df_movies_adj["adj_gross"] = df_movies_adj["gross"] * (258.811167 / df_movies_adj["CPI"])

# ROI calculation
df_movies_adj["ROI"] = ((df_movies_adj["adj_gross"] - df_movies_adj["adj_budget"]) / df_movies_adj["adj_budget"]) * 100

# Score sizing with exaggerated disparity
df_movies_adj["score_sized"] = df_movies_adj["score"] ** 3

In [5]:
fig = px.scatter(df_movies_adj, x="adj_budget", y="adj_gross",
                 hover_name="name",
                 labels={"adj_budget": "Adjusted Budget", "adj_gross": "Adjusted Gross"},
                 template="plotly_dark",
                 hover_data=["budget", "gross", "company", "director", "star", "score", "released", "ROI"],
                 opacity=0.7)

fig.update_layout(
    title="Movie Budgets vs Gross (Adjusted for Inflation)",
    xaxis_title="Budget ($USD)",
    yaxis_title="Gross ($USD)",
    showlegend=False,
    xaxis_range=[df_movies_adj["adj_budget"].min(), df_movies_adj["adj_budget"].max() + 5e6],
    yaxis_range=[df_movies_adj["adj_gross"].min(), df_movies_adj["adj_gross"].max() + 75e6]
)

pio.renderers.default = "browser"
fig.show()

In [7]:
fig = px.scatter(df_movies_adj, x="adj_budget", y="adj_gross",
                 hover_name="name",
                 labels={"adj_budget": "Adjusted Budget", "adj_gross": "Adjusted Gross"},
                 template="plotly_dark",
                 hover_data=["budget", "gross", "company", "director", "star"],
                 color="ROI",
                 color_continuous_scale="Thermal",
                 range_color=[0, df_movies_adj["ROI"].quantile(0.90)],
                 animation_frame="year",
                 size="score_sized",
                 size_max=20,
                 opacity=0.7)

fig.update_layout(
    title="Movie Budgets vs Gross (Adjusted for Inflation)",
    xaxis_title="Budget ($USD)",
    yaxis_title="Gross ($USD)",
    showlegend=False,
    xaxis_range=[df_movies_adj["adj_budget"].min(), df_movies_adj["adj_budget"].max()],
    yaxis_range=[df_movies_adj["adj_gross"].min(), df_movies_adj["adj_gross"].max() + 75e6]
)

pio.renderers.default = "browser"
fig.show()